# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-azeemi/fatima-flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



> The Core Objective: Translate our model's probability scores into an actionable, prioritized queue for the content team using the standard, established reason codes (STALE_HIGH_IMPRESSIONS, STALE_LOW_IMPRESSIONS) and action labels (REFRESH_CONTENT, REVIEW_STALE, MONITOR).

> Reason Codes & Action Mapping:

> STALE_HIGH_IMPRESSIONS: Content age $\ge 90$ days with high impressions ($\ge 500$). Action: REFRESH_CONTENT (Prioritize protecting organic traffic share).

> STALE_LOW_IMPRESSIONS: Content age $\ge 180$ days with low impressions. Action: REVIEW_STALE (Audit for keyword alignment or consolidation).


> LOW_PRIORITY: Newer or stable content. Action: MONITOR.

In [2]:
import pandas as pd
import numpy as np
import os, sys
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Setup repository path safely for Google Colab or local runs
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Prepare features and model training to generate live risk scores
feature_cols = ['word_count', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr', 'search_volume']
X = df[feature_cols].copy()
y = (df['trend_direction'] == 'down').astype(int)

# Impute and fit Random Forest model to compute risk probabilities
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_imputed, y)

# Assign continuous risk probability score
df['risk_probability'] = rf.predict_proba(X_imputed)[:, 1]

# Map Action and Reason Codes strictly matching previous W04/ML-07 rules
def assign_action_playbook(row):
    if row['content_age_days'] >= 90 and row['impressions_90d'] >= 500:
        return 'REFRESH_CONTENT', 'STALE_HIGH_IMPRESSIONS'
    elif row['content_age_days'] >= 180:
        return 'REVIEW_STALE', 'STALE_LOW_IMPRESSIONS'
    else:
        return 'MONITOR', 'LOW_PRIORITY'

res = df.apply(assign_action_playbook, axis=1)
df['action_label'] = [r[0] for r in res]
df['reason_code'] = [r[1] for r in res]

# Sort queue by highest risk probability first
df_queue = df.sort_values(by='risk_probability', ascending=False).reset_index(drop=True)

print(f"=== RANKED ACTION QUEUE GENERATED (Total Rows: {len(df_queue):,}) ===")
print(df_queue[['content_id', 'risk_probability', 'action_label', 'reason_code', 'impressions_90d', 'content_age_days']].head(5).to_string(index=False))

=== RANKED ACTION QUEUE GENERATED (Total Rows: 30,000) ===
          content_id  risk_probability    action_label            reason_code  impressions_90d  content_age_days
content_0d9c0ed65840          0.821473         MONITOR           LOW_PRIORITY              382               165
content_ee6c6b09c17d          0.819443 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS              916               104
content_ea733ea77a8f          0.817800         MONITOR           LOW_PRIORITY              397                96
content_cca1e559cffd          0.816818 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS             1339               104
content_d9e4b523c0ce          0.816708 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS              580               104


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*



> Intended Users: Content managers, SEO editors, and publishing leads responsible for large-scale content libraries (>10,000 pages).

> Intended Use: To prioritize weekly editorial workflows by flagging pages with high statistical risk of traffic decline, shifting teams from a reactive firefighter model to a proactive maintenance schedule.
Where It Stops Being Valid:

> 1. It cannot predict sudden macro algorithm rollouts, core updates, or external penalty actions.


> 2. It does not replace qualitative editorial judgment regarding brand voice, user intent shifts, or seasonal query trends.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify queue summary distribution across action labels and reason codes
print("=== ACTION QUEUE DISTRIBUTION SUMMARY ===")
summary_action = df_queue['action_label'].value_counts()
print(summary_action.to_string())

print("\n=== REASON CODE DISTRIBUTION SUMMARY ===")
summary_reason = df_queue['reason_code'].value_counts()
print(summary_reason.to_string())

=== ACTION QUEUE DISTRIBUTION SUMMARY ===
action_label
REFRESH_CONTENT    16726
REVIEW_STALE        8057
MONITOR             5217

=== REASON CODE DISTRIBUTION SUMMARY ===
reason_code
STALE_HIGH_IMPRESSIONS    16726
STALE_LOW_IMPRESSIONS      8057
LOW_PRIORITY               5217


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

> Human Review Requirements: Before altering or rewriting any flagged page, an editor must manually verify:

> 1. Whether the page serves a timeless evergreen intent where age is irrelevant.

> 2. Whether traffic drops are tied to temporary seasonal holidays rather than content degradation.

> The No-Go List (What Should NEVER Be Automated):

> 1. Automatic mass-rewriting or publishing of AI-generated content updates without human editorial oversight.

> 2. Automatic URL deprecation, canonical redirection, or page deletion based solely on model risk probabilities.**bold text**

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Sample check of top high-risk flags requiring manual review inspection
top_risk_sample = df_queue.head(10)[['content_id', 'risk_probability', 'action_label', 'reason_code', 'impressions_90d']]
print("=== TOP 10 HIGH-RISK QUEUE SAMPLE FOR HUMAN REVIEW ===")
print(top_risk_sample.to_string(index=False))

=== TOP 10 HIGH-RISK QUEUE SAMPLE FOR HUMAN REVIEW ===
          content_id  risk_probability    action_label            reason_code  impressions_90d
content_0d9c0ed65840          0.821473         MONITOR           LOW_PRIORITY              382
content_ee6c6b09c17d          0.819443 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS              916
content_ea733ea77a8f          0.817800         MONITOR           LOW_PRIORITY              397
content_cca1e559cffd          0.816818 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS             1339
content_d9e4b523c0ce          0.816708 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS              580
content_6a2b337eb3be          0.816489 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS              690
content_eedda1271288          0.816275 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS              955
content_345ddb5d3d57          0.815124 REFRESH_CONTENT STALE_HIGH_IMPRESSIONS              881
content_f552433bab3c          0.814427         MONITOR           LOW_PRIORITY             

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

> Data Drift Triggers: If the baseline distribution of monthly search impressions or average content age shifts by more than 25% due to platform migration or vertical expansion.

> Performance Degradation Trigger: If weekly Precision@50 drops significantly below baseline validation thresholds, indicating that flagged items no longer correspond to actual organic decline.

> Retrain Frequency: Re-train the Random Forest model on rolling 90-day windows on a quarterly basis.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Sanity check for monitoring metrics baseline
print("=== MONITORING BASELINE STATS ===")
print(f"Baseline Mean Risk Probability: {df_queue['risk_probability'].mean():.4f}")
print(f"Baseline Median Impressions: {df_queue['impressions_90d'].median():,.0f}")
print(f"Total Pages Tracked: {len(df_queue):,}")

=== MONITORING BASELINE STATS ===
Baseline Mean Risk Probability: 0.5418
Baseline Median Impressions: 731
Total Pages Tracked: 30,000


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

# Create outputs directory safely
os.makedirs('work/outputs', exist_ok=True)

# Define output columns for paper consumption
export_cols = [
    'content_id', 'client_id', 'risk_probability',
    'action_label', 'reason_code', 'impressions_90d',
    'content_age_days', 'word_count'
]
output_path = 'work/outputs/action_queue.csv'

df_queue[export_cols].to_csv(output_path, index=False)

print(f"✅ Successfully exported ranked action queue to '{output_path}'")
print(f"Exported rows count: {len(df_queue):,}")

✅ Successfully exported ranked action queue to 'work/outputs/action_queue.csv'
Exported rows count: 30,000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.